# 🎨 AbiArt — Kaggle GPU Backend (Automatic1111 SD WebUI API)

### ⚡ Configuration requise dans Kaggle (panneau à droite) :
1. **Accelerator** → `GPU T4 x2` (ou `P100`)
2. **Internet** → `ON`
3. Cliquez sur **Run All** (ou exécutez les cellules 1 à 5 dans l'ordre).
4. À la fin, copiez l'URL Cloudflare affichée (`https://xxxx.trycloudflare.com`) et collez-la dans AbiArt → Backend Kaggle → Connect.

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 1 — VÉRIFICATION DU GPU
# ══════════════════════════════════════════════════
import subprocess, sys
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode != 0:
    print('\u274c ERREUR : Aucun GPU détecté !')
    print('    → Allez dans le panneau à droite → Accelerator → GPU T4 x2')
    print('    → Activez aussi Internet → ON')
    print('    → Puis relancez.')
    sys.exit(1)
else:
    print(r.stdout.split('\n')[0])
    print('\u2713 GPU actif et détecté !')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 2 — INSTALLATION COMPLETE & REPOSITORIES AUTO-FIX
# ══════════════════════════════════════════════════
import os, sys, shutil, subprocess

WEBUI = '/kaggle/working/stable-diffusion-webui'
os.environ['STABLE_DIFFUSION_REPO'] = 'https://github.com/w-e-w/stablediffusion.git'
os.environ['GENERATIVE_MODELS_REPO'] = 'https://github.com/Stability-AI/generative-models.git'
os.environ['K_DIFFUSION_REPO'] = 'https://github.com/crowsonkb/k-diffusion.git'

# 1. Cloner Automatic1111 si absent
if not os.path.exists(WEBUI):
    print('Clonage d\'Automatic1111...')
    !git clone https://github.com/AUTOMATIC1111/stable-diffusion-webui.git {WEBUI}

%cd {WEBUI}

# 2. Créer sitecustomize.py pour blinder l'import de pytorch_lightning.utilities.distributed
with open(os.path.join(WEBUI, 'sitecustomize.py'), 'w') as f:
    f.write('''
import sys
try:
    import pytorch_lightning.utilities as plu
    if not hasattr(plu, 'distributed'):
        import types
        dist = types.ModuleType('pytorch_lightning.utilities.distributed')
        try:
            from pytorch_lightning.utilities.rank_zero import rank_zero_only
            dist.rank_zero_only = rank_zero_only
        except Exception:
            dist.rank_zero_only = lambda fn: fn
        plu.distributed = dist
        sys.modules['pytorch_lightning.utilities.distributed'] = dist
except Exception:
    pass
''')

# 3. Patcher requirements_versions.txt (uniquement Pillow et scikit-image pour compatibilité wheel)
!sed -i 's/Pillow==9.5.0/Pillow>=10.0.0/g' requirements_versions.txt
!sed -i 's/scikit-image==0.21.0/scikit-image>=0.21.0/g' requirements_versions.txt

# 4. Installation des paquets pip
print('\n[1/4] Installation des requirements A1111...')
!pip install -r requirements_versions.txt --prefer-binary 2>&1 | tail -5

print('\n[2/4] Installation de CLIP et open-clip-torch...')
!pip install --no-build-isolation https://github.com/openai/CLIP/archive/d50d76daa670286dd6cacf3bcd80b5e4823fc8e1.zip 2>&1 | tail -3
!pip install -q "open-clip-torch>=2.20.0" "pytorch-lightning==1.9.4" xformers ultralytics mediapipe

# 5. Cloner proprement TOUS les sous-dépôts avec leurs commits exacts (SANS shallow clone)
print('\n[3/4] Configuration des sous-dépôts (generative-models, stablediffusion, k-diffusion)...')
os.makedirs('repositories', exist_ok=True)

repos = [
    {
        'name': 'generative-models',
        'url': 'https://github.com/Stability-AI/generative-models.git',
        'commit': '45c443b316737a4ab6e40413d7794a7f5657c19f'
    },
    {
        'name': 'stable-diffusion-stability-ai',
        'url': 'https://github.com/w-e-w/stablediffusion.git',
        'commit': 'cf1d67a6fd5ea1aa600c4df58e5b47da45f6bdbf'
    },
    {
        'name': 'k-diffusion',
        'url': 'https://github.com/crowsonkb/k-diffusion.git',
        'commit': 'ab527a9a1a6def93310065275e7a96570c675317'
    },
    {
        'name': 'BLIP',
        'url': 'https://github.com/salesforce/BLIP.git',
        'commit': '48211a158d637db1138166184f897fba9b340bd3'
    },
    {
        'name': 'CodeFormer',
        'url': 'https://github.com/sczhou/CodeFormer.git',
        'commit': 'c5ed3084e33425522f6774fdcea4f0c439704f37'
    },
    {
        'name': 'stable-diffusion-webui-assets',
        'url': 'https://github.com/AUTOMATIC1111/stable-diffusion-webui-assets.git',
        'commit': None
    }
]

for r in repos:
    dest = os.path.join(WEBUI, 'repositories', r['name'])
    need_clone = True
    if os.path.exists(dest):
        if r['commit']:
            # Vérifier si le commit est déjà présent et valide
            chk = subprocess.run(['git', '-C', dest, 'checkout', r['commit']], capture_output=True, text=True)
            if chk.returncode == 0:
                print(f"  \u2713 {r['name']} déjà prêt ({r['commit'][:7]})")
                need_clone = False
            else:
                print(f"  \u26a0\ufe0f Nettoyage de l'ancien clone shallow de {r['name']}...")
                shutil.rmtree(dest, ignore_errors=True)
        else:
            need_clone = False
            print(f"  \u2713 {r['name']} présent")
    
    if need_clone:
        print(f"  Clonage complet de {r['name']}...")
        subprocess.run(['git', 'clone', r['url'], dest], check=True)
        if r['commit']:
            subprocess.run(['git', '-C', dest, 'checkout', r['commit']], check=True)
            print(f"  \u2713 {r['name']} verrouillé sur {r['commit'][:7]}")

# 6. Extensions utiles
print('\n[4/4] Vérification des extensions...')
!git clone --depth 1 https://github.com/Bing-su/adetailer.git extensions/adetailer 2>/dev/null || true
!git clone --depth 1 https://github.com/Mikubill/sd-webui-controlnet.git extensions/sd-webui-controlnet 2>/dev/null || true

print('\n\u2713 Environnement et sous-dépôts entièrement configurés !')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 3 — AUDIT COMPLET DE TOUS LES MODULES
# ══════════════════════════════════════════════════
import importlib, sys, os

WEBUI = '/kaggle/working/stable-diffusion-webui'
for sub in ['stable-diffusion-stability-ai', 'generative-models', 'k-diffusion', 'BLIP']:
    p = os.path.join(WEBUI, 'repositories', sub)
    if p not in sys.path:
        sys.path.insert(0, p)

modules_a_tester = [
    ('torch',                          'PyTorch'),
    ('torchvision',                    'TorchVision'),
    ('xformers',                       'xFormers'),
    ('PIL',                            'Pillow'),
    ('skimage',                        'scikit-image'),
    ('clip',                           'CLIP (OpenAI)'),
    ('open_clip',                      'open_clip (SDXL Conditioning)'),
    ('pytorch_lightning',              'PyTorch Lightning'),
    ('pytorch_lightning.utilities.distributed', 'PL distributed (rank_zero_only)'),
    ('gradio',                         'Gradio'),
    ('transformers',                   'Transformers'),
    ('safetensors',                    'Safetensors'),
    ('accelerate',                     'Accelerate'),
    ('einops',                         'Einops'),
    ('kornia',                         'Kornia'),
    ('omegaconf',                      'OmegaConf'),
    ('ldm.models.diffusion.ddpm',      'LDM (SD 1.5/2.1 backend)'),
    ('sgm.modules.encoders.modules',   'SGM (SDXL generative-models)'),
    ('k_diffusion',                    'k-diffusion samplers')
]

print('Vérification des imports...')
ok, fail = 0, 0
for mod, label in modules_a_tester:
    try:
        importlib.import_module(mod)
        print(f'  \u2713 {label}')
        ok += 1
    except Exception as e:
        print(f'  \u274c {label} → ERREUR : {e}')
        fail += 1

print(f'\n{ok} réussis / {fail} échecs')
if fail == 0:
    print('\u2713✓ TOUS LES MODULES SONT 100% FONCTIONNELS ! Vous pouvez lancer la suite.')
else:
    print('\u26a0\ufe0f Des dépendances nécessitent une attention. Vérifiez ci-dessus.')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 4 — TÉLÉCHARGEMENT DES MODÈLES (Animagine XL 3.1 + SDXL VAE)
# ══════════════════════════════════════════════════
import os
WEBUI = '/kaggle/working/stable-diffusion-webui'
ckpt_dir = f'{WEBUI}/models/Stable-diffusion'
vae_dir  = f'{WEBUI}/models/VAE'

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(vae_dir, exist_ok=True)

if not os.path.exists(f'{ckpt_dir}/animagine-xl-3.1.safetensors'):
    print('Téléchargement Animagine XL 3.1...')
    !wget -q --show-progress -c https://huggingface.co/cagliostrolab/animagine-xl-3.1/resolve/main/animagine-xl-3.1.safetensors -O {ckpt_dir}/animagine-xl-3.1.safetensors
else:
    print('\u2713 Animagine XL 3.1 déjà présent')

if not os.path.exists(f'{vae_dir}/sdxl_vae.safetensors'):
    print('Téléchargement SDXL VAE...')
    !wget -q --show-progress -c https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors -O {vae_dir}/sdxl_vae.safetensors
else:
    print('\u2713 SDXL VAE déjà présent')

print('\u2713 Modèles prêts !')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 5 — TUNNEL CLOUDFLARE + LANCEMENT API DIRECT
# ══════════════════════════════════════════════════
import os, subprocess, time, re, threading

WEBUI = '/kaggle/working/stable-diffusion-webui'

# 1. Installer Cloudflare Tunnel
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1

# 2. Démarrer le tunnel Cloudflare
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

def afficher_url():
    for line in iter(tunnel.stderr.readline, ''):
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m:
            print('\n' + '='*60)
            print('🎨 URL KAGGLE POUR ABIART :')
            print(f'   >>>  {m.group(0)}  <<<')
            print('='*60)
            print('Collez cette URL dans AbiArt → Backend Kaggle → Connect\n')
            break

threading.Thread(target=afficher_url, daemon=True).start()
time.sleep(3)

# 3. Lancement immédiat sans aucun check git/pip perturbateur
%cd {WEBUI}
!python launch.py \
    --skip-prepare-environment \
    --skip-install \
    --skip-torch-cuda-test \
    --skip-python-version-check \
    --enable-insecure-extension-access \
    --no-half-vae \
    --xformers \
    --api \
    --listen \
    --port 7860